In [3]:
import os
import boto3
from dotenv import load_dotenv
import pandas as pd

In [9]:
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")


In [10]:
load_dotenv("myenv.env")

True

In [18]:
client = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
MODEL_ID = "amazon.nova-micro-v1:0"

In [19]:
question = """
A shop sells pencils for ₹5 each.
Rahul buys 8 pencils and pays with ₹50.
How much change should he receive?
"""

In [20]:
zero_shot = f"""
Answer the following question:

{question}
"""

In [21]:
few_shot = f"""
Example:

Question:
A pen costs ₹10.
A person buys 3 pens and pays ₹50.
How much change does he get?

Answer:
Cost = 3 × 10 = ₹30
Change = 50 - 30 = ₹20

Now solve:

{question}
"""

In [22]:
cot = f"""
Solve the problem step by step and explain your reasoning.

{question}
"""

In [23]:
def get_response(prompt):

    response = client.converse(
        modelId=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [
                    {"text": prompt}
                ]
            }
        ]
    )

    return response["output"]["message"]["content"][0]["text"]

In [25]:

results = []

for name, prompt in [
    ("Zero-Shot", zero_shot),
    ("Few-Shot", few_shot),
    ("Chain-of-Thought", cot)
]:

    output = get_response(prompt)

    results.append({
        "Technique": name,
        "Output": output
    })

df = pd.DataFrame(results)

# Print complete outputs
for _, row in df.iterrows():
    print("=" * 50)
    print("Technique:", row["Technique"])
    print("=" * 50)
    print(row["Output"])
    print("\n")

Technique: Zero-Shot
- **Calculate the total cost of 8 pencils**:  Each pencil costs ₹5, so 8 pencils cost 8 * ₹5 = ₹40.

- **Determine the amount paid by Rahul**:  Rahul pays with ₹50.

- **Calculate the change Rahul should receive**:  Subtract the total cost from the amount paid: ₹50 - ₹40 = ₹10.

The answer is **₹10**


Technique: Few-Shot
To determine how much change Rahul should receive, we need to follow these steps:

1. Calculate the total cost of the pencils.
2. Subtract the total cost from the amount Rahul paid.

**Step 1: Calculate the total cost of the pencils**

Each pencil costs ₹5, and Rahul buys 8 pencils.
So, the total cost is:
\[ 8 \times 5 = ₹40 \]

**Step 2: Subtract the total cost from the amount Rahul paid**

Rahul pays ₹50.
The change he should receive is:
\[ 50 - 40 = ₹10 \]

Therefore, Rahul should receive ₹10 in change.


Technique: Chain-of-Thought
To determine how much change Rahul should receive, we need to follow these steps:

1. **Calculate the total cost 